# 0에서 AlexNet까지: 퍼셉트론, MLP, CNN의 출발점

이 노트북은 단순히 코드를 실행하는 예제가 아니라, **왜 다음 구조가 필요해졌는지**를 그림과 실험으로 확인하는 작은 교과서입니다.

학습 순서: 입력과 가중합 → 계단 함수 → 퍼셉트론 학습 규칙 → 선형 분리 → XOR 한계 → 은닉층과 MLP → 이미지의 공간 구조 → AlexNet

> 역사적으로 CNN은 AlexNet 이전에도 존재했습니다. LeNet 계열이 대표적입니다. 다만 2012년 AlexNet은 GPU 학습, ReLU, Dropout, 데이터 증강을 결합해 대규모 이미지 인식에서 딥 CNN의 가능성을 널리 입증한 전환점입니다.

## 학습 목표

이 노트북을 마치면 다음 질문에 직접 답할 수 있습니다.

1. 퍼셉트론 한 개는 입력을 어떻게 0 또는 1로 바꾸는가?
2. 오답 하나가 가중치와 결정경계를 어떻게 움직이는가?
3. AND는 배우지만 XOR은 배우지 못하는 이유는 무엇인가?
4. 은닉층과 활성화 함수가 표현력을 어떻게 늘리는가?
5. MLP가 이미지를 펼쳐 버릴 때 잃는 정보는 무엇인가?
6. AlexNet은 현대 CNN의 흐름에서 어떤 전환점이었는가?

In [ ]:
# 이 노트북에서 사용할 작은 학습 알고리즘과 시각화 함수입니다.
# 모든 예제는 CPU에서 수 초 안에 실행되도록 작게 설계했습니다.
import torch
import matplotlib.pyplot as plt

from cifar10_lab import (
    ClassicPerceptron,
    create_model,
    detect_environment,
    evaluate_model_detailed,
    get_lab_paths,
    load_cifar10_data,
    make_logic_gate,
    train_model,
    train_tiny_mlp,
)
from cifar10_lab.foundation_visualization import (
    plot_activation_functions,
    plot_hidden_representation,
    plot_mlp_learning_history,
    plot_perceptron_anatomy,
    plot_perceptron_learning,
    plot_xor_comparison,
)
from cifar10_lab.visualization import plot_test_results, plot_training_history

torch.manual_seed(42)
print('PyTorch:', torch.__version__)
print('Runtime:', detect_environment())

## 1. 퍼셉트론 한 개의 계산

입력 x₁, x₂ 각각에 가중치 w₁, w₂를 곱하고 편향 b를 더합니다.

**z = x₁w₁ + x₂w₂ + b**

고전적인 퍼셉트론은 z가 0 이상이면 1, 아니면 0을 출력합니다. 가중치는 각 입력의 영향력이고, 편향은 판단 기준을 좌우로 이동시키는 값입니다. 아래 셀의 sample, weights, bias를 바꿔 화살표마다 전달되는 값과 최종 판단이 어떻게 달라지는지 확인하세요.

In [ ]:
# 입력 두 개, 가중치 두 개, 편향 하나를 직접 바꿔 보세요.
sample = (1.0, 0.5)
weights = (0.8, -0.4)
bias = 0.1

plot_perceptron_anatomy(sample=sample, weights=weights, bias=bias)
plt.show()

## 2. 활성화 함수가 필요한 이유

계단 함수는 판단을 설명하기 쉽지만 미분할 수 없는 지점이 있어 경사하강법에 적합하지 않습니다. Sigmoid와 Tanh는 부드럽지만 큰 절댓값에서 기울기가 작아질 수 있습니다. ReLU는 양수 영역에서 기울기를 유지하여 깊은 CNN 학습을 크게 실용화했습니다. AlexNet의 중요한 선택 중 하나도 ReLU였습니다.

In [ ]:
plot_activation_functions()
plt.show()

## 3. AND 게이트를 오답 수정으로 학습하기

퍼셉트론 학습 규칙은 단순합니다. 예측이 틀리면 오차에 입력과 학습률을 곱해 가중치를 고칩니다.

**w ← w + 학습률 × (정답 - 예측) × x**

**b ← b + 학습률 × (정답 - 예측)**

AND의 정답은 (1, 1)일 때만 1입니다. 네 점은 직선 하나로 나눌 수 있으므로 단일 퍼셉트론이 학습할 수 있습니다.

In [ ]:
# 1) 네 가지 입력과 AND 정답을 준비합니다.
and_x, and_y = make_logic_gate('AND')

# 2) 모든 가중치가 0인 고전적 퍼셉트론을 만듭니다.
and_model = ClassicPerceptron(input_features=2, learning_rate=0.1)

# 3) 같은 네 표본을 10회 반복해서 보여 줍니다.
# history에는 표본 하나를 볼 때마다 바뀐 가중치, 편향, 오차가 기록됩니다.
and_history = and_model.fit(and_x, and_y, epochs=10)

print('입력       정답  최종예측')
for inputs, target, prediction in zip(and_x, and_y, and_model.predict(and_x)):
    print(f'{inputs.tolist()}    {int(target)}       {int(prediction)}')
print('최종 가중치:', and_model.weights.tolist())
print('최종 편향:', and_model.bias)

In [ ]:
# 실제로 오답이 발생해 파라미터가 갱신된 순간만 골라서 보여 줍니다.
# 각 패널에서 직선의 위치가 달라지는 것이 학습 그 자체입니다.
plot_perceptron_learning(and_x, and_y, and_history)
plt.show()

## 4. 단일 퍼셉트론의 한계: XOR

XOR은 두 입력이 서로 다를 때만 1입니다. 같은 클래스의 점이 대각선 방향에 놓이므로 직선 하나로 두 클래스를 분리할 수 없습니다. 학습 횟수를 늘려도 구조 자체가 표현할 수 없는 문제는 해결되지 않습니다.

MLP는 여러 퍼셉트론을 은닉층에 배치하고 비선형 활성화를 적용합니다. 각 은닉 뉴런이 서로 다른 경계를 만들고, 출력층이 이를 조합하면 XOR 같은 비선형 패턴을 표현할 수 있습니다.

In [ ]:
xor_x, xor_y = make_logic_gate('XOR')

# 단일 퍼셉트론: 계속 학습해도 네 점을 모두 맞히지 못합니다.
xor_perceptron = ClassicPerceptron(learning_rate=0.1)
xor_perceptron.fit(xor_x, xor_y, epochs=30)

# 작은 MLP: 2차원 입력 → Tanh 은닉층 → 출력층 구조입니다.
xor_mlp, xor_history = train_tiny_mlp(
    xor_x, xor_y, epochs=500, learning_rate=0.05, seed=42
)

print('퍼셉트론 예측:', xor_perceptron.predict(xor_x).tolist())
with torch.no_grad():
    mlp_predictions = (torch.sigmoid(xor_mlp(xor_x)) >= 0.5).long()
print('MLP 예측:      ', mlp_predictions.tolist())
print('정답:          ', xor_y.long().tolist())
print('MLP 최종 정확도:', xor_history['accuracy'][-1], '%')

In [ ]:
# 손실이 내려가고 정확도가 올라가는 과정은 경사하강법이 파라미터를 찾는 흔적입니다.
plot_mlp_learning_history(xor_history)
plt.show()

In [ ]:
# 왼쪽은 직선 하나, 오른쪽은 여러 은닉 뉴런이 합쳐 만든 비선형 영역입니다.
plot_xor_comparison(xor_perceptron, xor_mlp, xor_x, xor_y)
plt.show()

In [ ]:
# 은닉층은 단순히 분류만 하는 것이 아니라 입력을 분류하기 쉬운 좌표로 다시 표현합니다.
# 오른쪽 그림에서 h1, h2는 사람이 지정한 특징이 아니라 학습으로 만들어진 특징입니다.
plot_hidden_representation(xor_mlp, xor_x, xor_y)
plt.show()

## 5. 논리 게이트에서 CIFAR-10 이미지로

| 단계 | 입력 처리 | 얻는 능력 | 남는 한계 |
|---|---|---|---|
| 퍼셉트론 | 32×32×3 픽셀을 3,072개 값으로 펼침 | 클래스별 선형 경계 | 비선형 패턴과 위치 관계를 표현하기 어려움 |
| MLP | 펼친 픽셀을 여러 은닉층에 통과 | 비선형 경계와 학습된 특징 | 이웃 픽셀이라는 공간 정보를 구조적으로 활용하지 못함 |
| AlexNet | 합성곱으로 작은 영역을 반복 탐색 | 위치에 공유되는 특징, 계층적 시각 패턴 | 큰 모델과 많은 연산량 |
| 현대 CNN | 잔차 연결, 효율적 블록 등 | 더 깊고 안정적인 학습 | 설계별 정확도·효율 trade-off |
| Vision Transformer | 이미지를 패치 토큰으로 처리 | 전역 관계와 확장성 | 데이터와 계산량 요구가 클 수 있음 |

핵심 차이는 파라미터 수만이 아닙니다. CNN은 가까운 픽셀끼리 관계가 있다는 이미지의 구조를 합성곱이라는 가정으로 모델에 넣습니다. 이를 **귀납적 편향**이라고 합니다.

In [ ]:
# 같은 CIFAR-10 입력이 세 시대의 모델을 모두 통과하는지 확인합니다.
# 파라미터 수가 커진다고 항상 효율적이거나 정확한 것은 아닙니다.
sample_images = torch.randn(2, 3, 32, 32)

for model_id in ('perceptron', 'mlp', 'alexnet', 'resnet18'):
    model = create_model(model_id, num_classes=10, image_size=32).eval()
    parameter_count = sum(parameter.numel() for parameter in model.parameters())
    with torch.no_grad():
        output = model(sample_images)
    print(f'{model_id:12s} | parameters={parameter_count:>11,} | output={tuple(output.shape)}')

## 6. 같은 파이프라인에서 CIFAR-10을 직접 학습하기

아래 셀은 기본적으로 선형 퍼셉트론을 2,048개 이미지로 1 epoch 학습합니다. 데이터가 없으면 자동 다운로드됩니다. CIFAR_MODEL_ID를 mlp 또는 alexnet으로 바꾸면 데이터 분할과 평가 조건은 그대로 유지되므로 구조의 차이를 비교할 수 있습니다.

빠른 실행 결과는 교육용 흐름 확인을 위한 것이며 최종 성능 비교가 아닙니다. 공정한 비교에는 동일한 epoch, seed, 증강, optimizer뿐 아니라 모델마다 적절한 학습률 탐색도 필요합니다.

In [ ]:
# 처음에는 perceptron으로 전체 흐름을 빠르게 확인하세요.
# 이후 mlp, alexnet, resnet18 순서로 바꾸어 차이를 관찰할 수 있습니다.
CIFAR_MODEL_ID = 'perceptron'
QUICK_EPOCHS = 1

runtime = detect_environment()
paths = get_lab_paths(create=True)
trainloader, valloader, testloader, classes = load_cifar10_data(
    batch_size=64,
    seed=42,
    num_workers=runtime.num_workers,
    pin_memory=runtime.pin_memory,
    max_train_samples=2048,
    max_val_samples=512,
    max_test_samples=512,
)

model = create_model(CIFAR_MODEL_ID, num_classes=10, image_size=32).to(runtime.device)
history = train_model(
    model,
    trainloader,
    valloader,
    runtime.device,
    epochs=QUICK_EPOCHS,
    learning_rate=0.001,
    model_id=CIFAR_MODEL_ID,
    weight_dir=paths.checkpoints_dir / 'foundations',
)
result = evaluate_model_detailed(model, testloader, runtime.device, num_classes=10)
print(f'Test accuracy: {result["accuracy"]:.2f}%')

In [ ]:
# 학습 곡선은 최적화가 진행되는 과정, confusion matrix는 클래스별 실수를 보여 줍니다.
plot_training_history(history)
plot_test_results(result, classes)
plt.show()

## 7. AlexNet에서 다음 시대로

AlexNet을 관찰할 때는 정확도 하나보다 다음 연결을 보세요.

- 퍼셉트론의 가중합은 합성곱 필터 안에서도 반복됩니다.
- MLP의 ReLU와 Dropout은 AlexNet에서도 핵심 구성 요소입니다.
- 합성곱은 같은 필터를 이미지 전체에 공유해 MLP보다 공간 구조를 효율적으로 사용합니다.
- 이후 VGG는 반복적 구조, ResNet은 잔차 연결, MobileNet은 효율성, ViT는 패치 기반 전역 관계를 발전시켰습니다.

다음 단계에서는 main.ipynb에서 동일한 데이터 분할과 평가 방법으로 여러 백본을 비교해 보세요.